In [ ]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# 사용할 제공자의 키만 있으면 됨 — 설정 여부와 앞 2글자만 확인용으로 출력 (키 전체는 출력하지 않음)
for name, key in (("GROQ_API_KEY", GROQ_API_KEY), ("OPENAI_API_KEY", OPENAI_API_KEY)):
    status = f"설정됨 ({key[:2]}...)" if key else "없음"
    print(f"{name}: {status}")

#### LLM 설정 (Groq / OpenAI 선택)

- **Groq**와 **OpenAI** 두 가지 모델을 모두 사용할 수 있습니다. (Groq는 OpenAI 호환 API라서 `ChatOpenAI`로 연결)
- 아래 셀의 `PROVIDER` 값(`"groq"` 또는 `"openai"`)만 바꾸면 이후 모든 예제의 모델이 바뀝니다. 값을 바꾼 뒤에는 이 셀부터 다시 실행하세요.
- 사용할 제공자의 키(`GROQ_API_KEY` 또는 `OPENAI_API_KEY`)가 `.env`에 있어야 합니다. OpenAI를 선택하면 사용 요금이 발생합니다.

In [ ]:
from langchain_openai import ChatOpenAI

# 사용할 LLM 제공자 선택: "groq" 또는 "openai" — 이 값만 바꾸면 이후 모든 셀의 모델이 바뀜
PROVIDER = "groq"

# 제공자별 설정 (Groq는 OpenAI 호환 API라서 base_url만 다르고, OpenAI는 base_url 생략 시 기본 엔드포인트 사용)
LLM_CONFIGS = {
    "groq": {
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",  # Groq API 엔드포인트
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "api_key": OPENAI_API_KEY,
        "base_url": None,
        "model": "gpt-4o-mini",  # 대안: "gpt-4o"
    },
}
MAX_RETRIES = 5  # 503 등 일시 장애 자동 재시도 횟수


def get_llm(temperature: float = 0.5, seed: int | None = None, provider: str | None = None, **kwargs) -> ChatOpenAI:
    """선택한 제공자(기본값: PROVIDER)의 ChatOpenAI 인스턴스를 생성합니다.
    kwargs: ChatOpenAI 에 그대로 전달할 추가 옵션 (예: timeout=60)"""
    provider = provider or PROVIDER
    if provider not in LLM_CONFIGS:
        raise ValueError(f"지원하지 않는 제공자: {provider} (사용 가능: {list(LLM_CONFIGS)})")
    config = LLM_CONFIGS[provider]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {provider.upper()}_API_KEY 가 없습니다.")
    return ChatOpenAI(
        **config,
        temperature=temperature,
        seed=seed,  # 값을 고정하면 재현성이 향상됨 (완전히 동일한 결과는 보장되지 않음)
        max_retries=MAX_RETRIES,
        **kwargs,
    )

#### 1. CommaSeparatedListOutputParser

In [ ]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
import csv

# 콤마로 구분된 리스트 출력 파서 초기화
output_parser = CommaSeparatedListOutputParser()

# 출력 형식 지침 가져오기
format_instructions = output_parser.get_format_instructions()
print(format_instructions)

In [ ]:

# 프롬프트 템플릿 설정
prompt = PromptTemplate(
    template="List five {subject}.\n{format_instructions}",
    input_variables=["subject"],
    partial_variables={"format_instructions": format_instructions},
)

# LLM 설정 (제공자/키/모델명은 공통 설정 셀의 get_llm 사용)
llm = get_llm(temperature=0.7)
print(f"{PROVIDER} / {llm.model_name}")


In [ ]:

# 프롬프트, 모델, 출력 파서를 연결하여 체인 생성
chain = prompt | llm | output_parser

# "AI 관련 기술"에 대한 체인 호출 실행
result = chain.invoke({"subject": "AI 관련 기술"})

# 쉼표로 구분된 리스트 출력
print("AI 관련 기술 목록:")
print(result)

In [ ]:

# 결과 활용 예시: CSV 파일로 저장
csv_filename = "../data/ai_technologies.csv"
with open(csv_filename, "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["AI 기술"])  # 헤더 추가
    for item in result:
        writer.writerow([item])

print(f" '{csv_filename}' 파일로 저장 완료!")


#### 2. JsonOutputParser

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_openai import ChatOpenAI
import json

# JSON 출력 파서 초기화
parser = JsonOutputParser()

# 프롬프트 템플릿을 설정합니다.
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 과학 분야 전문가 AI입니다. 질문에 대해 체계적이고 간결한 답변을 JSON 형식으로 제공하세요. "
                   "JSON의 키 이름은 질문에서 지정한 영문 키를 그대로 사용하고, 값은 반드시 한국어로 작성하세요."),
        ("user", "#Format: {format_instructions}\n\n#Question: {question}"),
    ]
)

# JSON 출력 형식 지침을 프롬프트에 적용
prompt = prompt.partial(format_instructions=parser.get_format_instructions())


# 프롬프트, 모델, 출력 파서를 연결하는 체인 생성
chain = prompt | llm | parser

# 질문 설정 (우주 탐사 관련 질문)
question = "최근 10년간 진행된 주요 우주 탐사 미션 3가지를 알려주세요. \
각 미션의 이름은 `mission_name`에, 목표는 `goal`에, 주관 기관은 `agency`에 담아 주세요."

# 체인 실행 및 JSON 응답 받기
response = chain.invoke({"question": question})
print(type(response))

# JSON 데이터 출력
print(json.dumps(response, indent=4, ensure_ascii=False))


#### 3. PandasDataFrameOutputParser
* langchain 1.2 버전에서는 PandasDataFrameOutputParser 가 삭제 되었음

In [ ]:
# import pandas as pd
# from langchain_core.output_parsers import PandasDataFrameOutputParser
# from langchain_core.prompts import PromptTemplate
# from langchain_openai import ChatOpenAI
# import re

# # Titanic 데이터셋 로드
# df = pd.read_csv('../data/titanic.csv')

# # ChatOpenAI 모델 초기화
# #llm = ChatOpenAI(temperature=0, model_name="gpt-3.5-turbo")

# # Pandas DataFrame Output Parser 설정
# parser = PandasDataFrameOutputParser(dataframe=df)

# # 형식 지침 출력
# format_instructions = parser.get_format_instructions()
# print("Format Instructions:\n", format_instructions)

# # 프롬프트 템플릿 설정
# prompt = PromptTemplate(
#     template=""" 
#     You are a helpful assistant that interacts with a Pandas DataFrame.
#     The DataFrame contains the following columns: {columns}.
    
#     Your task is to answer the user's query by generating a command in the following format:
#     {format_instructions}
    
#     User Query: {query}    
#     """,
#     input_variables=["query"],
#     partial_variables={
#         "format_instructions": format_instructions,
#         "columns": ", ".join(df.columns)
#     },
# )

# # 체인 생성
# chain = prompt | llm | parser

# # 모델 응답 받기
# try:
#     # **Name 열을 표시하십시오.**
#     print('Name 컬럼 출력')
#     df_query = "Show the Name column"

#     parser_output = chain.invoke({"query": df_query})
#     print(type(parser_output))
#     print(parser_output)

#     # **첫번째 행을 표시하십시오.**
#     print('첫번째 행 출력')
#     df_query2 = "Show first row"

#     parser_output2 = chain.invoke({"query": df_query2})
#     print(parser_output2)

#     #Please tell me the average value of the Fare column.
#     print('Fare 컬럼의 평균값 출력')
#     df_query3 = "Show me the average value of the Fare column."

#     parser_output3 = chain.invoke({"query": df_query3})
#     print(parser_output3)

# except Exception as e:
#     print(f"오류 발생: {e}")

# # 프롬프트 템플릿 설정
# prompt = PromptTemplate(
#     template=""" 
#     You are a helpful assistant that interacts with a Pandas DataFrame.
#     The DataFrame contains the following columns: {columns}.
    
#     Your task is to answer the user's query by generating a command in the following format:
#     {format_instructions}
    
#     User Query: {query}    
#     """,
#     input_variables=["query"],
#     partial_variables={
#         "format_instructions": format_instructions,
#         "columns": ", ".join(df.columns)
#     },
# )

# # 체인 생성
# chain = prompt | llm | parser

# # 모델 응답 받기
# try:
#     # **Name 열을 표시하십시오.**
#     print('Name 컬럼 출력')
#     df_query = "Show the Name column"

#     parser_output = chain.invoke({"query": df_query})
#     print(type(parser_output))
#     print(parser_output)

#     # **첫번째 행을 표시하십시오.**
#     print('첫번째 행 출력')
#     df_query2 = "Show first row"

#     parser_output2 = chain.invoke({"query": df_query2})
#     print(parser_output2)

#     #Please tell me the average value of the Fare column.
#     print('Fare 컬럼의 평균값 출력')
#     df_query3 = "Show me the average value of the Fare column."

#     parser_output3 = chain.invoke({"query": df_query3})
#     print(parser_output3)

# except Exception as e:
#     print(f"오류 발생: {e}")


#### PydanticOutputParser를 사용하여 다시 수정한 예제

In [ ]:
import pandas as pd
import re
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# Titanic 데이터셋 로드
df = pd.read_csv('../data/titanic.csv')

# Pydantic 모델 정의 (pandas 명령어 구조화)
class PandasCommand(BaseModel):
    """Pandas DataFrame에서 실행할 명령어"""
    command: str = Field(..., description="실행할 pandas 명령어 예: df['Name'], df.iloc[0], df['Fare'].mean()")

# LLM 초기화 (제공자/키/모델명은 공통 설정 셀의 get_llm 사용)
llm = get_llm(temperature=0.7)

# PydanticOutputParser 설정
parser = PydanticOutputParser(pydantic_object=PandasCommand)

# 형식 지침 출력
format_instructions = parser.get_format_instructions()
print("Format Instructions:\n", format_instructions)
print("-" * 50)

# 프롬프트 템플릿 설정 (ChatPromptTemplate 사용 권장)
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a pandas DataFrame expert for the Titanic dataset.
    
Columns: {columns}
    
User query에 맞는 **pandas 명령어만** 생성하세요. 다른 설명 없음."""),
    ("user", "{query}\n\n{format_instructions}")
])

# 입력 변수 부분 설정
prompt = prompt.partial(
    columns=", ".join(df.columns),
    format_instructions=format_instructions
)

# 체인 생성
chain = prompt | llm | parser

# 안전한 pandas 명령 실행 함수
def safe_execute_pandas(df, command: str):
    """허용된 pandas 명령어만 안전하게 실행"""
    safe_commands = {
        r"df\['[^']*'\]": lambda: eval(command, {"df": df}, {}),
        r'df\["[^"]*"\]': lambda: eval(command, {"df": df}, {}),
        r"df\.iloc\[\d+\]": lambda: eval(command, {"df": df}, {}),
        r"df\['[^']*'\]\.mean\(\)": lambda: eval(command, {"df": df}, {}),
        r"df\['[^']*'\]\.head\(\)": lambda: eval(command, {"df": df}, {}),
        r"df\.head\(\)": lambda: eval(command, {"df": df}, {}),
        r"df\.shape": lambda: eval(command, {"df": df}, {}),
    }
    
    command = command.strip()
    for pattern, executor in safe_commands.items():
        if re.match(pattern, command):
            try:
                return executor()
            except Exception as e:
                return f"실행 오류: {e}"
    return "허용되지 않은 명령어입니다."

# 테스트 실행
queries = [
    ("Name 컬럼 출력", "Show the Name column"),
    ("첫번째 행 출력", "Show first row"),
    ("Fare 컬럼 평균값 출력", "Show me the average value of the Fare column.")
]

for title, query in queries:
    print(f"\n{title}")
    print("=" * 40)
    
    try:
        result = chain.invoke({"query": query})
        print("생성된 명령어:", result.command)
        print("실행 결과:")
        output = safe_execute_pandas(df, result.command)
        print(output)
        print(type(output))
        
    except Exception as e:
        print(f"오류 발생: {e}")
        print(type(e))



In [ ]:
import pandas as pd
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# LLM 초기화 (제공자/키/모델명은 공통 설정 셀의 get_llm 사용)
llm = get_llm(temperature=0, seed=42, timeout=60)  # seed 고정으로 재현성 향상, timeout 60초


In [ ]:

# Pydantic 모델 정의 (테이블 데이터 구조화)
class TableData(BaseModel):
    """테이블 행을 나타내는 딕셔너리 리스트"""
    data: list[dict] = Field(..., description="테이블 행 데이터 (딕셔너리 리스트)")

# PydanticOutputParser 설정
parser = PydanticOutputParser(pydantic_object=TableData)

# ChatPromptTemplate 설정 (최신 LCEL 스타일)
prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 테이블 형태의 데이터를 생성하는 AI 어시스턴트입니다.
    매번 **동일한 데이터**를 정확히 동일한 순서로 반환하세요.
    반드시 다음 JSON 스키마를 정확히 따라야 합니다:
    {format_instructions}
    
    데이터는 현실적이고 일관성 있게 생성하세요. (실제 통계가 아닌 예시용 가상 데이터입니다.)"""),
    ("user", "{query}")
])

# 부분 변수 설정
prompt = prompt.partial(format_instructions=parser.get_format_instructions())

# 체인 생성
chain = prompt | llm | parser

# DataFrame 생성 함수
def generate_dataframe(user_query: str, expected_columns: list[str] | None = None) -> pd.DataFrame:
    """LLM으로 테이블 데이터를 생성해 DataFrame으로 반환합니다. 실패 시 빈 DataFrame을 반환합니다."""
    print(" 쿼리 처리 중:", user_query[:50] + "...")
    try:
        # 모델 호출
        result = chain.invoke({"query": user_query})
        print(" 생성된 JSON 데이터:", result)

        # DataFrame으로 변환
        df = pd.DataFrame(result.data)

        # 요청한 컬럼과 다르면 실패 처리
        if expected_columns and set(df.columns) != set(expected_columns):
            print(f" 컬럼 불일치: 기대 {expected_columns} / 실제 {list(df.columns)}")
            return pd.DataFrame()

        print("\n 생성된 DataFrame:")
        print(df)
        print(f"\n 데이터 형태: {df.shape} ({len(df.columns)} columns)")
        return df

    except Exception as e:
        print(f" 오류 발생: {type(e).__name__}: {e}")
        return pd.DataFrame()


In [ ]:

# 실행 테스트
print("2024년 상반기 서울 아파트 평균 매매 가격 데이터 생성")
print("=" * 60)

df_seoul_housing = generate_dataframe(
    "2024년 상반기 서울 아파트 평균 매매가격 데이터셋을 생성해주세요. "
    "컬럼: 구(District), 평균가격_억원(Average_Price_100M_KRW), 거래건수(Transactions), "
    "전년동기대비증감률_퍼센트(YoY_Change_Percent). "
    "서울의 25개 구에 대한 현실적인 데이터를 생성해주세요.",
    expected_columns=["District", "Average_Price_100M_KRW", "Transactions", "YoY_Change_Percent"],
)

print("\n" + "=" * 60)
print("최종 결과:")
print(df_seoul_housing)

In [ ]:
df_seoul_housing

In [ ]:
print('2024년 서울 지하철역별 유동 인구 데이터')
# [예제 2] 2024년 서울 지하철역별 유동 인구 데이터
df_seoul_subway = generate_dataframe(
    #"Generate a dataset of the top 10 busiest subway stations in Seoul in 2024 with columns: Station Name, Line Number, Daily Passenger Volume, and Weekday vs Weekend Ratio."
    "2024년 서울 지하철 승객 이용량이 가장 많은 상위 10개 역의 데이터셋을 생성해주세요. \
        컬럼은 다음과 같습니다: 역명(Station_Name), 호선(Line_Number),\
              일평균승객수_만명(Daily_Passengers_10K), 평일대주말비율(Weekday_Weekend_Ratio).\
                  실제 서울 지하철의 주요 역들을 기반으로 현실적인 데이터를 생성해주세요.",
    expected_columns=["Station_Name", "Line_Number", "Daily_Passengers_10K", "Weekday_Weekend_Ratio"],
)

#if df_seoul_subway is not None:
df_seoul_subway.head()


In [ ]:
df_seoul_subway

In [ ]:
print('한국 5대 편의점 브랜드별 2024년 매출 및 점포 수')
# [예제 3] 한국 5대 편의점 브랜드별 2024년 매출 및 점포 수
df_korean_convenience_stores = generate_dataframe(
    #"Create a dataset of the top 5 convenience store brands in Korea in 2024 with columns: Brand Name, Number of Stores, Total Revenue (in billion KRW), and Market Share (%)."
    "2024년 한국의 편의점 브랜드 상위 5개사 데이터셋을 생성해주세요.\
          컬럼은 다음과 같습니다: 브랜드명(Brand_Name), 점포수(Store_Count), \
            총매출_조원(Revenue_Trillion_KRW), 시장점유율_퍼센트(Market_Share_Percent).\
                  CU, GS25, 세븐일레븐, 이마트24, 미니스톱 등 실제 한국 편의점 브랜드를 기반으로 현실적인 데이터를 생성해주세요.",
    expected_columns=["Brand_Name", "Store_Count", "Revenue_Trillion_KRW", "Market_Share_Percent"],
)


In [ ]:

df_korean_convenience_stores